In [1]:
import duckdb
import polars as pl
import numpy as np
import pendulum
from pathlib import Path

data_dir = Path("../data/lastfm/listening")
db = duckdb.read_json(data_dir/"*.jsonl")
df = db.pl()

df = df.with_columns(
    pl.from_epoch(
        pl.col("date_played_unix"), time_unit="s"
    ).alias("track_played_utc")
)

print("Raw data:", df.shape)

df = df.unique("date_played_unix") # Not so exact with which duplicate gets removed
print("Duplicate timestamps removed:", df.unique("date_played_unix").shape)
df = df.with_columns(
    artist_name = pl.col("artist_name").str.to_lowercase(),
    track_name = pl.col("track_name").str.to_lowercase(),
    album_name = pl.col("album_name").str.to_lowercase()
)
df.head()

Raw data: (110513, 8)
Duplicate timestamps removed: (108066, 8)


artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc
str,str,str,str,str,str,i64,datetime[μs]
"""alvedon""","""""","""hurry up mixtape""","""""","""weargue.""","""""",1730490004,2024-11-01 19:40:04
"""grouper""","""7e571e09-887e-4544-b993-eb75e7…","""the man who died in his boat""","""0001130f-cb3a-49b1-9685-a2a734…","""being her shadow""","""8bcb1d02-89ce-4869-abe6-2449c5…",1731952995,2024-11-18 18:03:15
"""madvillain""","""4e024037-14b7-4aea-99ad-c6ace6…","""madvillainy""","""264702af-dd10-46b5-83ef-741955…","""do not fire!""","""""",1677573015,2023-02-28 08:30:15
"""clams casino""","""772a0eb3-fe31-4a08-b583-690296…","""instrumental mixtape 4 sampler""","""""","""worth it""","""ef2a502c-b4ae-46c6-bda1-4a30c7…",1634739012,2021-10-20 14:10:12
"""teen suicide""","""a63a5c27-aae4-40b3-86c7-527496…","""dc snuff film / waste yrself""","""0403574e-b166-4aa1-b27e-ab4d9c…","""benzo""","""1091257d-0aa7-44b5-852a-a7fc00…",1631354647,2021-09-11 10:04:07


## Your "anthem" — song played the most in a single day

In [2]:
(
    df
    .group_by(
        pl.col("track_played_utc").dt.date(), 
        pl.col("artist_name"), 
        pl.col("track_name")
    )
    .agg(
        n_daily_scrobbles = pl.len(),
    )
    .with_columns(
        rank = pl.col("n_daily_scrobbles").rank(method="ordinal", descending=True).over(
            pl.col("track_played_utc")
        )
    )
    .filter(pl.col("rank") <= 3)
).sort("track_played_utc", descending=False) # .sort("n_daily_scrobbles", descending=True)


track_played_utc,artist_name,track_name,n_daily_scrobbles,rank
date,str,str,u32,u32
2021-05-20,"""aldn""","""i'm alright""",2,1
2021-05-20,"""sematary""","""i'm a sinner""",2,2
2021-05-20,"""bladee""","""lordship""",2,3
2021-05-21,"""bladee""","""mallwhore freeestyle""",2,2
2021-05-21,"""shelly""","""steeeam""",3,1
…,…,…,…,…
2026-03-24,"""grouper""","""poison tree""",2,2
2026-03-24,"""playboi carti""","""lean 4 real (feat. skepta)""",2,3
2026-03-25,"""daft punk""","""around the world""",1,2


## Unique artists

In [3]:
artists_2025 = (
    df
    .filter(pl.col("track_played_utc").dt.year() == 2025)
    .group_by(pl.col("artist_name"))
    .agg(
        scrobbles = pl.len()
    )
)

artists_else = (
    df
    .filter(pl.col("track_played_utc").dt.year() < 2025)
    .select(pl.col("artist_name").unique())
)

unique_artists_2025 = artists_2025.join(artists_else, "artist_name", how="anti")
unique_artists_2025.sort(pl.col("scrobbles"), descending=True)

artist_name,scrobbles
str,u32
"""ulla""",250
"""cameron winter""",152
"""elias rønnenfelt""",135
"""geese""",132
"""emile mosseri""",112
…,…
"""demdike stare""",1
"""daniel romano""",1
"""dayseeker""",1


In [4]:
def unique_artists(df, year:int=2025):

    if year == 2021:
        raise ValueError("No data before 2021 is available.")

    artists_year = (
        df
        .filter(pl.col("track_played_utc").dt.year() == year)
        .group_by(pl.col("artist_name"))
        .agg(
            scrobbles = pl.len()
        )
    )

    artists_else = (
        df
        .filter(pl.col("track_played_utc").dt.year() < year)
        .select(pl.col("artist_name").unique())
    )

    unique_artists_year = artists_year.join(artists_else, "artist_name", how="anti")
    return unique_artists_year.sort(pl.col("scrobbles"), descending=True)

In [5]:
unique_artists(df, 2025).head(10)

artist_name,scrobbles
str,u32
"""ulla""",250
"""cameron winter""",152
"""elias rønnenfelt""",135
"""geese""",132
"""emile mosseri""",112
"""ragnhild og""",107
"""fine""",107
"""dj_dave""",105
"""snuggle""",94
